In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [27]:
df = pd.read_csv(r"C:\Tata AI Hackathon\dataset\train.csv")

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1352 entries, 0 to 1351
Data columns (total 51 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   CoilID  1352 non-null   int64  
 1   X1      1352 non-null   float64
 2   X2      1352 non-null   float64
 3   X3      1352 non-null   float64
 4   X4      1352 non-null   float64
 5   X5      1352 non-null   float64
 6   X6      1352 non-null   float64
 7   X7      1352 non-null   float64
 8   X8      1351 non-null   float64
 9   X9      1352 non-null   float64
 10  X10     1346 non-null   float64
 11  X11     1352 non-null   float64
 12  X12     1352 non-null   float64
 13  X13     1352 non-null   float64
 14  X14     1352 non-null   float64
 15  X15     1192 non-null   float64
 16  X16     1346 non-null   float64
 17  X17     1352 non-null   float64
 18  X18     1352 non-null   float64
 19  X19     1352 non-null   float64
 20  X20     1352 non-null   float64
 21  X21     1351 non-null   float64
 22  

In [29]:
df.describe()

,CoilID,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X41,X42,X43,X44,X45,X46,X47,X48,X49,Y
count,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1351.000000,1352.000000,...,1352.000000,1321.000000,1352.000000,1352.000000,1352.000000,1352.000000,1352.000000,1339.000000,1352.000000,1352.000000
mean,833.515533,1028.915322,575.374555,538.372787,692.576938,649.010832,618.584573,529.241056,528.770054,461.699867,...,0.585886,0.011334,0.281290,0.091495,11.507361,0.002745,0.050913,0.007108,0.064400,0.048817
std,487.420566,108.505011,232.103882,135.288085,56.798904,35.398207,45.602831,46.263179,40.220201,40.600916,...,0.296115,0.017114,0.068455,0.073058,25.004864,0.006660,0.044617,0.012504,0.050905,0.215564
min,1.000000,235.252250,96.755492,124.150450,575.916250,559.272859,529.937396,439.221384,425.413251,343.110465,...,0.077352,0.000000,0.029599,0.000000,-82.672877,0.000000,0.016833,0.000000,0.000000,0.000000
25%,411.250000,1009.279089,405.533242,441.585514,622.213663,625.327743,583.363796,476.813303,520.192858,423.604248,...,0.365881,0.000380,0.244377,0.026073,-1.368777,0.000943,0.039864,0.000000,0.015624,0.000000
50%,824.500000,1071.978233,589.160841,548.035306,724.970407,661.170690,615.240491,547.648095,545.399528,482.412285,...,0.574328,0.002613,0.293164,0.076108,9.177752,0.001644,0.045829,0.001056,0.060336,0.000000
75%,1254.250000,1092.031100,725.766924,632.238363,734.453213,668.054192,659.564239,554.584166,556.334294,488.713960,...,0.797025,0.013933,0.334613,0.143156,24.196605,0.002275,0.051435,0.002679,0.106017,0.000000
max,1691.000000,1124.903234,1148.171484,1026.915778,755.983296,763.257466,742.725523,618.947910,575.312130,505.349388,...,1.377925,0.062637,0.409739,0.329108,120.169658,0.064545,0.424405,0.051150,0.249522,1.000000


In [30]:
print(df['Y'].value_counts())

Y
0.0    1286
1.0      66
Name: count, dtype: int64


In [31]:
# ── Imports (cleaned, no duplicates) ────────────────────────────────────────
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_curve
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from collections import defaultdict
import numpy as np

# ── Class ratio ──────────────────────────────────────────────────────────────
pos   = y.sum()
neg   = len(y) - pos
ratio = round(neg / pos, 2)
print(f"Positive (defect): {pos}, Negative: {neg}, Ratio: {ratio}")

# ── Models ───────────────────────────────────────────────────────────────────
models = {

    "RandomForest": RandomForestClassifier(
        n_estimators=500,
        class_weight='balanced',
        min_samples_leaf=2,
        random_state=42
    ),

    # HistGradientBoosting handles imbalance and missing values natively
    # does NOT go into the SMOTE pipeline — has its own pipeline below
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=300,
        class_weight='balanced',
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        scale_pos_weight=ratio,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,           # helps generalisation
        colsample_bytree=0.8,    # randomly samples features per tree
        random_state=42,
        eval_metric='aucpr',
        verbosity=0
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=300,
        scale_pos_weight=ratio,
        min_child_samples=5,
        num_leaves=31,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    ),

    "CatBoost": CatBoostClassifier(
        iterations=300,
        scale_pos_weight=ratio,
        learning_rate=0.05,
        depth=4,
        random_state=42,
        verbose=0
    )
}

# ── Models that should NOT use SMOTE ─────────────────────────────────────────
# HistGradientBoosting handles everything natively
no_smote = {"HistGradientBoosting"}

ImportError: cannot import name 'tarfile_extractall' from 'sklearn.utils.fixes' (c:\Users\johad\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\fixes.py)

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
# pipe = Pipeline([
#     ('imputer', SimpleImputer(strategy='median')),
#     # ('classifier', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42) )
#     ('classifier', GradientBoostingClassifier(n_estimators=200, random_state=42) )
# ])

In [ ]:
# ── SKF ──────────────────────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X = df.drop(columns=['CoilID', 'Y'])
y = df['Y']

Model_best_thresholds = defaultdict(list)
Model_results         = defaultdict(list)

target_recall = 0.90   # minimum recall you will accept

for model_name, model in models.items():
    print("-" * 50)

    # ── Build the right pipeline per model ───────────────────────────────────
    if model_name in no_smote:
        # HistGradientBoosting: no imputer needed, no SMOTE
        from sklearn.pipeline import Pipeline as SkPipeline
        pipe = SkPipeline([
            ('classifier', model)
        ])
    else:
        # All others: impute missing values, then SMOTE, then model
        pipe = ImbPipeline([
            ('imputer',    SimpleImputer(strategy='median')),
            ('smote',      SMOTE(
                                random_state=42,
                                k_neighbors=5        # reduce if you get errors
                           )),
            ('classifier', model)
        ])

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        pipe.fit(X_train, y_train)
        y_prob = pipe.predict_proba(X_val)[:, 1]

        precisions, recalls, thresholds = precision_recall_curve(y_val, y_prob)
        f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

        # ── Strategy: hit recall target first, then maximise precision ───────
        valid_mask = recalls[:-1] >= target_recall

        if valid_mask.any():
            valid_precisions = precisions[:-1][valid_mask]
            valid_thresholds = thresholds[valid_mask]
            valid_recalls    = recalls[:-1][valid_mask]
            valid_f1s        = f1_scores[:-1][valid_mask]

            best_idx       = np.argmax(valid_precisions)
            best_threshold = valid_thresholds[best_idx]
            best_recall    = valid_recalls[best_idx]
            best_precision = valid_precisions[best_idx]
            best_f1        = valid_f1s[best_idx]
            met_target     = True
        else:
            # Recall target not achievable — fall back to best F1
            best_idx       = np.argmax(f1_scores)
            best_threshold = thresholds[best_idx]
            best_recall    = recalls[best_idx]
            best_precision = precisions[best_idx]
            best_f1        = f1_scores[best_idx]
            met_target     = False

        Model_best_thresholds[model_name].append(best_threshold)
        Model_results[model_name].append({
            'fold'      : fold,
            'threshold' : best_threshold,
            'recall'    : best_recall,
            'precision' : best_precision,
            'f1'        : best_f1,
            'met_target': met_target
        })

        flag = "✅" if met_target else "⚠️ recall target missed, using best F1"
        print(f"  {flag} {model_name} | Fold {fold} | "
              f"threshold: {best_threshold:.4f} | "
              f"recall: {best_recall:.4f} | "
              f"precision: {best_precision:.4f} | "
              f"F1: {best_f1:.4f}")

--------------------------------------------------
Model: RandomFC, Fold 1 - Best threshold:0.2450, best F1 score metric--recall: 0.6154, precision: 0.8889, F1 score: 0.7273
Model: RandomFC, Fold 2 - Best threshold:0.0250, best F1 score metric--recall: 1.0000, precision: 0.1321, F1 score: 0.2333
Model: RandomFC, Fold 3 - Best threshold:0.1600, best F1 score metric--recall: 0.3846, precision: 0.2273, F1 score: 0.2857
Model: RandomFC, Fold 4 - Best threshold:0.1900, best F1 score metric--recall: 0.6154, precision: 0.4000, F1 score: 0.4848
Model: RandomFC, Fold 5 - Best threshold:0.2750, best F1 score metric--recall: 0.5385, precision: 0.7000, F1 score: 0.6087
--------------------------------------------------
Model: GBoostC, Fold 1 - Best threshold:0.0357, best F1 score metric--recall: 0.7692, precision: 0.4762, F1 score: 0.5882
Model: GBoostC, Fold 2 - Best threshold:0.0659, best F1 score metric--recall: 0.5000, precision: 0.3500, F1 score: 0.4118
Model: GBoostC, Fold 3 - Best threshold

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"{'Model':<25} {'Recall':>8} {'Precision':>10} {'F1':>8} "
      f"{'Avg Thresh':>12} {'Target Met':>12}")
print("=" * 60)

for model_name, results in Model_results.items():
    mean_recall     = np.mean([r['recall']     for r in results])
    mean_precision  = np.mean([r['precision']  for r in results])
    mean_f1         = np.mean([r['f1']         for r in results])
    mean_threshold  = np.mean([r['threshold']  for r in results])
    folds_met       = sum([r['met_target']     for r in results])

    print(f"{model_name:<25} {mean_recall:>8.3f} {mean_precision:>10.3f} "
          f"{mean_f1:>8.3f} {mean_threshold:>12.4f} "
          f"{folds_met}/5 folds")

RandomFC - Average Best Threshold: 0.1790
GBoostC - Average Best Threshold: 0.0769
XGBoostClassifier - Average Best Threshold: 0.2481
XGBoostRFClassifier - Average Best Threshold: 0.4270
LGBoostMClassifier - Average Best Threshold: 0.1272
CatBClassifier - Average Best Threshold: 0.2685
